# Professional ML Pipeline: FashionMNIST Classification (PyTorch)

## Pipeline Overview & Stages

My goal is to show:
- How to write clean, reusable code (no copy-paste)
- How to organize each stage of the ML lifecycle
- How to structure code so I can experiment and change things easily

**Pipeline stages:**
1. Stage 0: Setup and utility functions
2. Stage 1: Load data and preprocessing
3. Stage 2: Understand the data (features, labels, batches)
4. Stage 3: Build model, loss, and optimizer
5. Stage 4: Train
6. Stage 5: Evaluate on test set

**For interviews:** I explain each stage as a decision point in the ML lifetime.

## Stage 0 - ML Pipeline Infrastructure

### Big Picture

This entire notebook builds an **End-to-End ML Pipeline**:

```
Data  →  Model  →  Training  →  Evaluation
```

**Stage 0** means: We build the **reusable infrastructure ONCE**, then use it throughout.

### What Stage 0 Does

✔ **Define all building blocks** that we'll use in the rest of the pipeline
✔ **No training happens here** — just setup
✔ **Production-style design** — reusable, testable, clean

### Stage 0 Components

1. **Imports** — Load PyTorch, datasets, utilities
2. **Config** — Hyperparameters (batch_size, learning_rate, num_classes)
3. **Transform** — Convert images to tensors
4. **Data Loading** — Download FashionMNIST
5. **DataLoader** — Mini-batch iteration
6. **Model** — Feedforward neural network
7. **Loss Function** — CrossEntropyLoss for classification
8. **Optimizer** — Adam (adaptive gradient descent)
9. **Training Loop** — One epoch of training
10. **Evaluation** — Accuracy computation
11. **Debug Tool** — Sample inspection

### Key Pattern: Factory Functions

All functions follow this naming convention:

- `create_*` — Config and utilities
- `build_*` — Architecture
- `train_*` — Training logic
- `evaluate_*` — Evaluation metrics
- `inspect_*` — Debugging helpers

This is **professional ML code** — you'll see this pattern in production systems. 💼

In [14]:
"""
=============================================================================
STAGE 0: ML PIPELINE INFRASTRUCTURE
=============================================================================
This cell defines all reusable building blocks for the entire ML pipeline.
We build the infrastructure ONCE, then use it throughout the notebook.
=============================================================================
"""

# =============================================================================
# SECTION 1: IMPORTS (Core Libraries)
# =============================================================================
# torch              → PyTorch core tensor operations
# nn                 → Neural network layers (Linear, ReLU, etc.)
# DataLoader         → Mini-batch iteration utility
# datasets           → Pre-built datasets (FashionMNIST)
# transforms         → Data preprocessing (ToTensor, normalization, etc.)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =============================================================================
# SECTION 2: HYPERPARAMETERS & CONFIG
# =============================================================================
# These are the "knobs" we can tune:
# - batch_size: how many samples per mini-batch (larger = faster, needs more GPU memory)
# - learning_rate: controls how large parameter updates are (larger = faster learning, risk of overshooting)
# - num_classes: number of output categories (FashionMNIST has 10)
DEFAULT_DATA_ROOT = "data"
DEFAULT_BATCH_SIZE = 64
DEFAULT_LR = 1e-3  # 0.001 (Adam's default is 0.001, which is good)
NUM_CLASSES = 10   # FashionMNIST has 10 classes: T-shirt, Trouser, Pullover, etc.

# =============================================================================
# SECTION 3: DATA PREPROCESSING (Transform)
# =============================================================================
# Purpose: Convert raw PIL images → PyTorch tensors
# Why: Models only understand numbers, not image files
# What ToTensor() does:
#   - Image shape [H, W, C] → tensor with shape [C, H, W]
#   - Pixel values [0-255] → normalized to [0.0-1.0]
def create_transform():
    """Create the default image preprocessing pipeline."""
    return transforms.ToTensor()


# =============================================================================
# SECTION 4: DATASET LOADING (Data I/O)
# =============================================================================
# Purpose: Download and load FashionMNIST dataset
# Returns: (train_dataset, test_dataset)
# Key pattern: Transform is passed into dataset constructor (not applied manually later)
def load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=None):
    """Download and load FashionMNIST train/test splits with consistent preprocessing."""
    if transform is None:
        transform = create_transform()

    # Training split: 60,000 images (model learns from this)
    train_ds = datasets.FashionMNIST(
        root=data_root,
        train=True,
        download=True,
        transform=transform,
    )
    
    # Test split: 10,000 images (evaluate generalization on unseen data)
    test_ds = datasets.FashionMNIST(
        root=data_root,
        train=False,
        download=True,
        transform=transform,
    )
    return train_ds, test_ds


# =============================================================================
# SECTION 5: MINI-BATCH ITERATOR (DataLoader)
# =============================================================================
# Purpose: Convert dataset → mini-batches for efficient training
# shuffle=True  for training   (randomize order to reduce learning artifacts)
# shuffle=False for evaluation (deterministic order for reproducibility)
def create_loader(dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=False):
    """Wrap a dataset in a DataLoader for mini-batch iteration."""
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


# =============================================================================
# SECTION 6: NEURAL NETWORK MODEL (Architecture)
# =============================================================================
# Simple feedforward architecture:
#   Input (1, 28, 28)
#   → Flatten to 784-dim vector
#   → Dense layer to 128 neurons + ReLU activation
#   → Dense layer to 10 neurons (logits for each class)
#   → Output (10 classes)
# 
# Why simple? Keep focus on ML pipeline, not architectural complexity.
# Can easily swap with CNN or deeper model later without changing the pipeline.
def build_model(input_shape=(1, 28, 28), hidden_size=128, num_classes=NUM_CLASSES):
    """Build a simple feedforward classifier."""
    _, h, w = input_shape
    return nn.Sequential(
        nn.Flatten(),                              # (B, 1, 28, 28) → (B, 784)
        nn.Linear(h * w, hidden_size),             # (B, 784) → (B, 128)
        nn.ReLU(),                                 # Apply non-linearity
        nn.Linear(hidden_size, num_classes),       # (B, 128) → (B, 10) [logits]
    )


# =============================================================================
# SECTION 7: LOSS FUNCTION (Objective)
# =============================================================================
# CrossEntropyLoss is the standard for multi-class classification:
# - Expects raw logits (not probabilities) from model
# - Expects integer class indices (not one-hot encoded labels)
# - Internally handles softmax → log_softmax → NLLLoss
def create_loss_fn():
    """Create the classification loss function."""
    return nn.CrossEntropyLoss()


# =============================================================================
# SECTION 8: OPTIMIZER (Parameter Update Rule)
# =============================================================================
# Adam: Adaptive Moment Estimation
# - Adaptive learning rates for each parameter
# - Fast convergence, works well in practice
# - Industry standard for deep learning
def create_optimizer(model, lr=DEFAULT_LR):
    """Create an Adam optimizer for model parameters."""
    return torch.optim.Adam(model.parameters(), lr=lr)


# =============================================================================
# SECTION 9: TRAINING LOOP (One Epoch)
# =============================================================================
# One epoch = one full pass through all training data
# Steps in loop:
#   1. Set model to training mode (enables dropout, batch norm updates)
#   2. For each mini-batch:
#      a. Forward pass: compute predictions
#      b. Compute loss: measure error
#      c. Backward pass: compute gradients via backpropagation
#      d. Update step: gradient descent on parameters
#   3. Return average loss for the epoch
def train_one_epoch(model, loader, loss_fn, optimizer, device="cpu"):
    """Train the model for one complete epoch."""
    model.train()  # Set to training mode (important for dropout/batch norm)
    running_loss = 0.0

    for images, labels in loader:
        # Move data to device (GPU or CPU)
        images, labels = images.to(device), labels.to(device)
        
        # ===== FORWARD PASS =====
        outputs = model(images)  # Predictions (logits)
        loss = loss_fn(outputs, labels)  # Compute error
        
        # ===== BACKWARD PASS (Backpropagation) =====
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute new gradients via chain rule
        optimizer.step()       # Update parameters using gradients
        
        # ===== LOSS ACCUMULATION =====
        # Track total loss (weighted by batch size for proper averaging)
        running_loss += loss.item() * labels.size(0)

    # Return average loss over the entire dataset
    return running_loss / len(loader.dataset)


# =============================================================================
# SECTION 10: EVALUATION (Inference + Metrics)
# =============================================================================
# During evaluation:
# - No gradient computation needed (inference only)
# - No dropout, batch norm uses running statistics
# - Deterministic predictions
# Metric: Accuracy = (correct predictions) / (total samples)
def evaluate_accuracy(model, loader, device="cpu"):
    """Measure classification accuracy on a dataset."""
    model.eval()  # Set to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation for speed/memory
        for images, labels in loader:
            # Move data to device
            images, labels = images.to(device), labels.to(device)
            
            # ===== INFERENCE =====
            outputs = model(images)  # (B, 10) logits
            predicted = outputs.argmax(dim=1)  # (B,) predicted class indices
            
            # ===== ACCURACY COMPUTATION =====
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total if total else 0.0


# =============================================================================
# SECTION 11: DEBUG UTILITY (Data Inspection)
# =============================================================================
# Purpose: Inspect raw data sample for debugging
# Returns: Dictionary with image, label, shape, class name
# Use case: Verify data loading, class mapping, tensor shapes before training
def inspect_sample(dataset, index=0):
    """Peek at one raw sample from the dataset (for debugging and exploration)."""
    image, label = dataset[index]
    return {
        "image": image,
        "label": label,
        "shape": tuple(image.shape),
        "class_name": dataset.classes[label],
        "class_to_idx": dataset.class_to_idx,
    }


# =============================================================================
# END OF STAGE 0 INFRASTRUCTURE
# =============================================================================
# Summary:
# ✔ Config: All hyperparameters defined in one place
# ✔ Data: Loading and preprocessing functions
# ✔ Model: Neural network architecture
# ✔ Optimization: Loss function and optimizer
# ✔ Training: One-epoch training routine
# ✔ Evaluation: Accuracy computation
# ✔ Debugging: Sample inspection utility
#
# Pattern: FACTORY FUNCTIONS (create_*, build_*, train_*, evaluate_*)
# This is production-style ML code — reusable, testable, maintainable.
# =============================================================================

---


## Stage 1 - Data Loading & Preprocessing

### Goal

Load the FashionMNIST dataset and apply consistent preprocessing.

### What We Do

1. **Create the preprocessing pipeline** — Define how to transform images
2. **Download the dataset** — FashionMNIST (train + test splits)
3. **Apply transforms** — Convert PIL images → normalized tensors
4. **Inspect shapes** — Verify data is correct before training

### Why This Stage Matters

- **Preprocessing parity**: Train and test get IDENTICAL preprocessing (prevents evaluation bias)
- **Lazy loading**: Dataset doesn't load all images into RAM (important for large datasets)
- **Reproducibility**: Same transforms everywhere → same results always

### Stage 1 Components

1. **Transform Definition** — What preprocessing to apply
2. **Train Dataset** — Load training split (60,000 images)
3. **Test Dataset** — Load test split (10,000 images)

In [ ]:
# =============================================================================
# STAGE 1.0: CREATE PREPROCESSING TRANSFORM
# =============================================================================
# Purpose: Define the transformation pipeline for all images
# What it does: PIL Image → PyTorch Tensor with values in [0.0, 1.0]
# Why: Models only understand tensors, not image files
# =============================================================================

transform = create_transform()  # Returns: transforms.ToTensor()

print("Transform pipeline created:")
print(f"  Input:  PIL Image")
print(f"  Output: Torch Tensor with shape (C, H, W) and values in [0.0, 1.0]")

## Stage 1 Deep Dive - Why Preprocessing Matters

### The Transformation Pipeline

Raw images are **PIL Image objects** (from disk):
- Format: RGB or Grayscale pixel arrays
- Value range: [0, 255] (unsigned integers)
- Not compatible with PyTorch models

**After `transforms.ToTensor()`:**
- Format: PyTorch tensor
- Value range: [0.0, 1.0] (normalized floats)
- Shape: (Channels, Height, Width)
- **Ready for the model** ✔

### Why Normalization is Crucial

1. **Numerical Stability** — Small, normalized values prevent gradient explosion/vanishing
2. **Faster Convergence** — Optimization performs better on [0,1] scale
3. **Consistent Learning** — Same preprocessing across train/test ensures fair comparison

### Preprocessing Parity Pattern

**Critical Rule**: Train and test preprocessing MUST be identical.

```python
# ✔ CORRECT (preprocessing before split)
transform = create_transform()
train_ds = load_fashion_mnist(train=True, transform=transform)
test_ds = load_fashion_mnist(train=False, transform=transform)  # SAME transform

# ✗ WRONG (different preprocessing)
train_ds = load_fashion_mnist(train=True, transform=normalize_v1)
test_ds = load_fashion_mnist(train=False, transform=normalize_v2)  # Different!
```

This ensures **fair, reproducible evaluation**.

## Reusable Preprocessing Pattern

### Why We Pass Transform Into Dataset

Instead of preprocessing manually in random cells:

```python
# ✗ BAD: Manual preprocessing scattered around
for each_image in raw_images:
    tensor = transforms.ToTensor()(each_image)  # Apply here
    # ...later in notebook...
    tensor = transforms.ToTensor()(each_image)  # Apply again (inconsistent!)
```

We use **factory pattern**:

```python
# ✔ GOOD: Transform defined once, reused consistently
transform = create_transform()
dataset = Dataset(..., transform=transform)  # Transform attached at dataset level
```

### Benefits

1. **Single Source of Truth** — Transform logic lives in one place
2. **Easy Testing** — Change `create_transform()` → all datasets affected
3. **Scalability** — Later add augmentation without touching dataset code
4. **Interview Strength** — Shows understanding of design patterns 💼

### Real-World Usage

Production ML systems use this pattern everywhere — it's the industry standard.

---


## Stage 1 - Load Datasets

### Stage 1.1 - Load Training Dataset

The next code cell loads the training split using the shared transform.

What to mention in interview:

- this split is used to update model parameters
- transformation is attached at dataset level
- loading logic is reusable and parameterized

In [ ]:
# =============================================================================
# STAGE 1.1: LOAD TRAINING DATASET
# =============================================================================
# Purpose: Load the training split of FashionMNIST
# Key Points:
#   - Training set: 60,000 images
#   - Used to optimize model parameters
#   - Preprocessing applied: transform=create_transform()
#   - If dataset not cached: auto-download from internet
# =============================================================================

train_dataset, _ = load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=transform)

print("Training dataset loaded:")
print(f"  Size: {len(train_dataset)} images")
print(f"  Classes: {len(train_dataset.classes)}")
print(f"  Class names: {train_dataset.classes}")

### Stage 1.2 - Load Test Data

Test set gets the same preprocessing as training.

**Why:** If train and test have different preprocessing, my results will be wrong. I use exactly the same transformation.

In [ ]:
# =============================================================================
# STAGE 1.2: LOAD TEST DATASET
# =============================================================================
# Purpose: Load the test split of FashionMNIST
# Key Points:
#   - Test set: 10,000 images (unseen during training)
#   - Used to evaluate generalization ability
#   - SAME preprocessing: transform=create_transform() (critical!)
#   - If preprocessing differs → unfair evaluation
# =============================================================================

_, test_dataset = load_fashion_mnist(data_root=DEFAULT_DATA_ROOT, transform=transform)

print("Test dataset loaded:")
print(f"  Size: {len(test_dataset)} images")
print(f"  Classes: {len(test_dataset.classes)}")
print(f"  Preprocessing: Identical to training (fair evaluation guaranteed)")

### End of Stage 1

Now I have the data loaded and consistent preprocessing applied.

Next: understand what's inside the data.

## Stage 2 - Data Understanding & Validation

### Goal

**Verify that the data is correct** before training the model.

*"Trust, but verify." — Production ML Rule #1*

### What We Do

1. **Inspect class mapping** — Verify all 10 classes are loaded correctly
2. **Check sample dimensions** — Ensure tensors have expected shape
3. **Validate label encoding** — Confirm labels are integers [0-9]
4. **Test batch loading** — Verify DataLoader produces correct shapes

### Why This Stage Matters

**Catching data bugs early saves hours of debugging later.**

- Wrong shapes → Training crashes with cryptic error
- Misaligned labels → Model learns nonsense with no warning
- Class imbalance → Silent failure (model predicts majority class)

### Stage 2 Components

1. **Class Mapping** — FashionMNIST class labels and names
2. **Single Sample Inspection** — Look at raw data
3. **Batch Inspection** — Look at mini-batches
4. **Quick Checklist** — Before moving to Stage 3

In [ ]:
# =============================================================================
# STAGE 2.0: RECREATE TRANSFORM (For Clarity)
# =============================================================================
# Purpose: Recreate the transform at stage boundary for explicit clarity
# This helps readers understand data flow through each stage
# (In production, you'd avoid duplication, but for learning it's pedagogical)
# =============================================================================

transform = create_transform()
print("Stage 2: Data Validation & Inspection")
print(f"Transform: {transform}")

### Step 2.1: Inspect Class Mapping

FashionMNIST has two views of class information:

**`dataset.classes`** — List of class names in order:
```python
['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
```

**`dataset.class_to_idx`** — Dictionary mapping name to index:
```python
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, ...}
```

**Gold Rule**: Always check both to catch encoding bugs early! ✔

In [ ]:
# =============================================================================
# STAGE 2.1: INSPECT CLASS NAMES
# =============================================================================
# Extract the list of class names (ordered by index)
# =============================================================================

class_names = train_dataset.classes
print("Class names in order:")
for idx, name in enumerate(class_names):
    print(f"  {idx:2d}: {name}")

T-shirt/top


In [ ]:
# =============================================================================
# STAGE 2.2: INSPECT CLASS-TO-INDEX MAPPING
# =============================================================================
# Dictionary view: class names → indices
# This confirms bidirectional mapping is consistent
# =============================================================================

print("\nClass-to-Index mapping:")
print(train_dataset.class_to_idx)

{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


### Step 2.3: Inspect a Single Sample

Pull out ONE training sample and check:
- ✔ Tensor shape is (1, 28, 28) — correct!
- ✔ Label is an integer [0-9] — correct!
- ✔ Class name matches label — correct!
- ✔ No NaNs or infinities — correct!

This is the **first data validation you should do** in any ML project.

In [ ]:
# =============================================================================
# STAGE 2.3: INSPECT SINGLE SAMPLE
# =============================================================================
# Purpose: Validate one sample before batching (debugging is easier at n=1)
# =============================================================================

# Get the first training sample
image, label = train_dataset[0]

print(f"First training sample:")
print(f"  Image shape: {image.shape} (expect: [1, 28, 28])")
print(f"  Label (int): {label}")
print(f"  Class name: {class_names[label]}")
print(f"  Value range: [{image.min():.3f}, {image.max():.3f}] (expect: [0.0, 1.0])")
print(f"  Data type: {image.dtype} (expect: torch.float32)")
print(f"  No NaNs: {not image.isnan().any()} ✔")

image.shape= torch.Size([1, 28, 28]) class=
label= 9
class= Ankle boot


### Step 2.4: Inspect a Mini-Batch

Now test the DataLoader — it should return tensors with batch dimension:

Expected shape:
- Images: **(batch_size, 1, 28, 28)** — e.g., [64, 1, 28, 28]
- Labels: **(batch_size,)** — e.g., [64]

If this shape is wrong, **STOP and debug** before training! ✋

---

#### Load Training DataLoader and Inspect First Batch

The DataLoader wraps our dataset and provides mini-batches for training.

In [18]:
# =============================================================================
# STAGE 2.4: CREATE TRAINING DATALOADER
# =============================================================================
# Purpose: Create mini-batch iterator for training
# Key Points:
#   - batch_size=64: Stack 64 images per batch (GPU-efficient)
#   - shuffle=True: Random order (prevents overfitting to sequence)
#   - test_dataset used with shuffle=False (later)
# =============================================================================

train_loader = create_loader(
    train_dataset, 
    batch_size=DEFAULT_BATCH_SIZE, 
    shuffle=True
)

print("Training DataLoader created:")
print(f"  Total batches per epoch: {len(train_loader)}")
print(f"  Batch size: {DEFAULT_BATCH_SIZE}")
print(f"  Shuffle: True (random order each epoch)")


Training DataLoader created:
  Total batches per epoch: 938
  Batch size: 64
  Shuffle: True (random order each epoch)


In [23]:
# =============================================================================
# STAGE 2.5: CREATE TEST DATALOADER
# =============================================================================
# Purpose: Create mini-batch iterator for evaluation
# Key Points:
#   - shuffle=False: KEEP sequential order (consistent evaluation)
#   - batch_size can match training (or be different for memory)
# =============================================================================

test_loader = create_loader(
    test_dataset, 
    batch_size=DEFAULT_BATCH_SIZE, 
    shuffle=False  # Critical: no shuffling for evaluation
)

print("Test DataLoader created:")
print(f"  Total batches: {len(test_loader)}")
print(f"  Batch size: {DEFAULT_BATCH_SIZE}")
print(f"  Shuffle: False (keep order for evaluation)")


Test DataLoader created:
  Total batches: 157
  Batch size: 64
  Shuffle: False (keep order for evaluation)


### Understanding the Batch Shape

For `images.shape = [64, 1, 28, 28]`:
- `64` = number of images in this mini-batch
- `1` = grayscale (1 channel)
- `28 x 28` = image dimensions

For `labels.shape = [64]`:
- one class per image
- images[i] and labels[i] are paired

This confirms the data structure my model expects.

In [ ]:
# =============================================================================
# STAGE 2.6: INSPECT FIRST BATCH
# =============================================================================
# Purpose: Validate batch shape before training
# This confirms the data pipeline is correctly configured
# =============================================================================

# Get first batch
batch_images, batch_labels = next(iter(train_loader))

print("First batch from DataLoader:")
print(f"  Images shape: {batch_images.shape} (expect: [64, 1, 28, 28])")
print(f"  Labels shape: {batch_labels.shape} (expect: [64])")
print(f"  Image value range: [{batch_images.min():.3f}, {batch_images.max():.3f}]")
print(f"  Unique labels in batch: {batch_labels.unique()}")
print(f"✓ Batch validation PASSED - ready for training!")

Image shape: torch.Size([1, 28, 28])
Label: 9
Image tensor:
<class 'torch.Tensor'>
tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000

## Stage 3 - Model & Optimization Setup

### Goal

**Build the machine learning model** and configure its learning algorithm.

### What We Do

1. **Initialize model architecture** — Create neural network
2. **Attach loss function** — Define optimization target
3. **Create optimizer** — Configure gradient descent algorithm
4. **Send to device** — Move to GPU if available

### Why This Stage Matters

**The "secret recipe" lives here.** Small changes to:
- Layer sizes → Huge impact on model capacity
- Learning rate → Determines training speed (too high = divergence, too low = slow)
- Optimizer choice → Adam vs. SGD vs. others

**Interview Question**: "Why did you choose this optimizer?" → Answer should reference learning rate adaptation, momentum, etc.

### Stage 3 Components

1. **Model** — Neural network definition
2. **Loss Function** — Training objective
3. **Optimizer** — Parameter update algorithm
4. **Device Assignment** — CPU or GPU

### Step 3.1: Initialize Model

Define the neural network architecture and move it to the device (GPU or CPU).

---

### Loss Function Example

The next cell shows the expected format for `CrossEntropyLoss` and why I don't need one-hot labels here.

In [16]:
# =============================================================================
# STAGE 3.1: BUILD AND INITIALIZE MODEL
# =============================================================================
# Purpose: Create neural network and move to device
# Key Points:
#   - build_model() returns Sequential net: 784→128→10
#   - .to(device) moves weights to GPU if available (critical for speed!)
#   - device = 'cuda' if GPU available, else 'cpu'
# =============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model()
model = model.to(device)

print(f"Model initialized on device: {device}")
print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

Model initialized on device: cpu
Model architecture:
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=10, bias=True)
)

Total parameters: 101,770


### Step 3.2: Define Loss Function

CrossEntropyLoss is the standard choice for multi-class classification:
- Combines softmax (probability conversion) + log loss
- Expects raw logits (no softmax in model output)
- Compatible with 10-class classification

---


## Stage 3 - Build Model & Optimization

### Stage 3.1 - Create Training DataLoader

`train_loader` controls batching and randomization during training.

Using `shuffle=True` helps avoid fixed-order learning artifacts and improves robustness of stochastic optimization.

In [ ]:
# Create training loader for optimization stage.
train_loader = create_loader(train_dataset, batch_size=DEFAULT_BATCH_SIZE, shuffle=True)

### Stage 3.2 - Build Model

We initialize a reusable baseline network for FashionMNIST classification.

This stage isolates architecture creation from training logic, which keeps experimentation clean and maintainable.

### Architecture Notes

Input: tensor with shape `[1, 28, 28]`
Flatten: converts spatial image to a 784-dimensional vector
Hidden layer: learns compact intermediate features
Output layer: produces 10 logits for class decision

This architecture is intentionally simple to keep focus on pipeline quality and training mechanics.

This baseline is ideal for interviews because it demonstrates complete ML flow without architectural noise.

The same reusable pipeline can later host CNN or deeper models by only changing `build_model()`.

In [ ]:
# Build baseline model.
model = build_model()

### Stage 3.3 - Define Loss Function

In [19]:
# =============================================================================
# STAGE 3.2: CREATE LOSS FUNCTION
# =============================================================================

criterion = create_loss_fn()
print(f"Loss function: {criterion}")

Loss function: CrossEntropyLoss()


### Step 3.3: Create Optimizer

Adam (Adaptive Moment Estimation) is the go-to for modern deep learning:
- **Adaptive learning rates** — Different rates for different parameters
- **Momentum** — Helps escape local minima
- **Robust default** — Works well with minimal tuning

### Stage 3.4 - Define Optimizer

In [20]:
# =============================================================================
# STAGE 3.3: CREATE OPTIMIZER
# =============================================================================
# Purpose: Configure gradient descent algorithm
# Key Points:
#   - Adam optimizer: adaptive learning rates
#   - lr=0.001: learning rate (adjust if training diverges)
#   - model.parameters(): reference to all trainable weights
# =============================================================================

optimizer = create_optimizer(model, lr=DEFAULT_LR)
print(f"Optimizer: {optimizer}")
print(f"Learning rate: {DEFAULT_LR}")

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


## Stage 4 - Training

### Goal

**Optimize model parameters** by running the training loop.

### What We Do

1. **Forward pass** — Predict output: `model(batch_images) → logits`
2. **Compute loss** — Compare predictions to ground truth: `criterion(logits, labels)`
3. **Backward pass** — Compute gradients: `loss.backward()`
4. **Update weights** — Gradient descent step: `optimizer.step()`
5. **Repeat** — Run for N epochs until convergence

### Why This Stage Matters

**This is where the "learning" happens.**

- Each epoch → model gets slightly better
- Watching loss decrease → confidence that training works
- Early stopping → Prevent overfitting

### Stage 4 Components

1. **Multi-epoch training loop** — Repeat over training data
2. **Loss tracking** — Monitor convergence
3. **Validation during training** — Catch overfitting
4. **Model checkpoint** — Save best weights

### Step 4.1: Multi-Epoch Training Loop

## Stage 4 - Train the Model

In [21]:
# =============================================================================
# STAGE 4.1: TRAINING LOOP
# =============================================================================
# Purpose: Optimize model weights over multiple epochs
# Key Points:
#   - num_epochs=5: How many times to loop through training data
#   - model.train(): Enable dropout and batch normalization
#   - .backward(): Compute gradients
#   - optimizer.step(): Update weights based on gradients
# =============================================================================

num_epochs = 5
training_history = []

for epoch in range(num_epochs):
    # Enter training mode
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    # Loop through training batches
    for batch_images, batch_labels in train_loader:
        # Move data to device
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass: predict logits
        logits = model(batch_images)
        
        # Compute loss
        loss = criterion(logits, batch_labels)
        
        # Backward pass: compute gradients
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute new gradients
        
        # Update weights: gradient descent
        optimizer.step()
        
        # Track loss
        epoch_loss += loss.item()
        num_batches += 1
    
    # Log epoch results
    avg_loss = epoch_loss / num_batches
    training_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

print("✓ Training complete!")

Epoch 1/5 | Loss: 0.5430
Epoch 2/5 | Loss: 0.4035
Epoch 3/5 | Loss: 0.3632
Epoch 4/5 | Loss: 0.3361
Epoch 5/5 | Loss: 0.3148
✓ Training complete!


The training cell runs one reusable epoch routine with the following sequence:

1. switch model to training mode
2. iterate over mini-batches
3. run forward pass
4. compute loss
5. compute gradients via backpropagation
6. update parameters via optimizer

The printed loss is the epoch-level average and serves as a primary optimization signal.

---


## Stage 5 - Evaluation & Results

### Goal

**Measure how well the trained model generalizes** to new, unseen data.

### What We Do

1. **Set model to inference mode** — `model.eval()`
2. **Disable gradient computation** — `with torch.no_grad():`
3. **Process test batches** — Run through all test data
4. **Compute accuracy** — Count correct predictions
5. **Report results** — Display final metrics

### Why This Stage Matters

**Training accuracy is a lie.** The model memorizes training data!

Real question: **Does it work on NEW data?** → That's what evaluation answers.

- High train accuracy, low test accuracy → Overfitting
- Low train accuracy, low test accuracy → Underfitting
- High train AND test accuracy → ✓ Model generalizes well

### Stage 5 Components

1. **Test batch processing** — Inference mode with no gradient computation
2. **Accuracy computation** — Percentage of correct predictions
3. **Loss on test set** — How wrong the model is on new data
4. **Summary statistics** — Train vs. test comparison

In [24]:
# =============================================================================
# STAGE 5.1: EVALUATE ON TEST SET
# =============================================================================
# Purpose: Measure generalization to unseen data
# Key Points:
#   - model.eval(): Disable training-only features (dropout, batchnorm)
#   - torch.no_grad(): Don't compute gradients (save memory)
#   - True "test" accuracy only if test data unseen during training
# =============================================================================

model.eval()  # Set to evaluation mode
test_correct = 0
test_total = 0

# Disable gradient computation for efficiency
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        # Move data to device
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass (no gradients needed)
        logits = model(batch_images)
        
        # Get predictions from logits (argmax = most likely class)
        predictions = logits.argmax(dim=1)
        
        # Count correct predictions
        test_correct += (predictions == batch_labels).sum().item()
        test_total += batch_labels.size(0)

# Compute accuracy
test_accuracy = 100.0 * test_correct / test_total
print(f"\n{'='*50}")
print(f"TEST RESULTS")
print(f"{'='*50}")
print(f"Correct predictions: {test_correct} / {test_total}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"{'='*50}")


TEST RESULTS
Correct predictions: 8663 / 10000
Test Accuracy: 86.63%


## Summary - Complete ML Pipeline 🎯

### What You've Learned

This notebook demonstrates a **production-grade ML pipeline** for image classification:

**Stage 0** — Reusable infrastructure (all utilities in one place)
- Factory functions for transforms, dataloaders, model, loss, optimizer
- Single source of truth — easy to modify and reuse

**Stage 1** — Data Loading & Preprocessing
- FashionMNIST dataset (10 classes, 60K train / 10K test)
- PIL → Tensor conversion + normalization
- Proof: "Preprocessing parity" (train/test use same transform)

**Stage 2** — Data Validation
- Class mapping verification
- Single sample inspection
- Batch shape validation
- **Gold rule**: Never train without validating data first

**Stage 3** — Model & Optimization Setup
- Simple feedforward: 784 → 128 → 10
- CrossEntropyLoss (standard for multi-class)
- Adam optimizer (adaptive learning rates)

**Stage 4** — Training
- 5 epochs of gradient descent
- Forward → Loss → Backward → Update cycle
- Loss tracking (convergence monitoring)

**Stage 5** — Evaluation
- Test accuracy = ground truth performance
- model.eval() + torch.no_grad() (inference mode)
- Generalization check (train vs. test)

### Interview-Ready Takeaways

✔ **Modular design** — All utilities encapsulated (factory pattern)
✔ **Separation of concerns** — Each stage has one job
✔ **Reproducibility** — Same transforms for train/test
✔ **Production patterns** — Batch processing, device handling, model.eval()
✔ **Clear documentation** — PURPOSE headers for every section
✔ **Scalability** — Easy to add data augmentation, regularization, etc.

### Next Steps (If Time Allows)

1. Add dropout for regularization
2. Implement learning rate scheduling
3. Add data augmentation (rotations, flips)
4. Visualize training curves
5. Save/load model checkpoints